In [1]:
from pycozo.client import Client
import pandas as pd
from pycozo.client import Client
client = Client('rocksdb', 'snbsf1.db')

In [2]:
res = client.run("?[n]:= *reified_node[n]")
res

,n
0,35184372090193
1,35184372090194
2,35184372090195
3,35184372090196
4,35184372090197
...,...
410194,35184372500387
410195,35184372500388
410196,35184372500389
410197,35184372500390


In [9]:
rn_df = pd.DataFrame(client.run("?[id_rn] := *reified_node[id_rn]"))
max_id = int(rn_df["id_rn"].max())
NEXT_RN = max_id + 1
NEXT_RN

35184372282716

In [2]:
import pandas as pd
from math import ceil

CHUNK_SIZE = 10000  

#Charger les données existantes dans des df
edge_df        = pd.DataFrame(client.run('?[id_e, ns, nd] := *edge[id_e, ns, nd]'))
edge_label_df  = pd.DataFrame(client.run('?[id_e, ln] := *edge_label[id_e, ln]'))
node_label_df  = pd.DataFrame(client.run('?[id_n, l] := *node_label[id_n, l]'))
reified_node_df= pd.DataFrame(client.run('?[id_rn] := *reified_node[id_rn]'))
n_con_node_df  = pd.DataFrame(client.run('?[id_rn, id_n] := *n_composed_of_node[id_rn, id_n]'))
n_con_edge_df  = pd.DataFrame(client.run('?[id_rn, id_e] := *n_composed_of_edge[id_rn, id_e]'))
node_prop_df   = pd.DataFrame(client.run('?[id_n, key, val] := *node_prop[id_n, key, val]'))

#Filtrer les arêtes hasMember
has_member_df = edge_label_df[edge_label_df['ln'] == 'hasMember'].merge(edge_df, on='id_e')

#Grouper par forum
forums = has_member_df.groupby('ns')  # ici ns = forum id

#initialiser des IDs réifiés
next_rn = reified_node_df['id_rn'].max() + 1 

new_reified_nodes = []
new_node_labels   = []
new_n_con_nodes   = []
new_n_con_edges   = []
new_node_props    = []

for forum_id, group in forums:
    rn_id = next_rn
    next_rn += 1
    
    #Reified node 
    new_reified_nodes.append([rn_id])
    
    # Node label 
    new_node_labels.append([rn_id, "forumMembership"])
    
    #Composed of edges (all hasMember edges for this forum) 
    for edge_id in group['id_e']: # ici id_e sont les ids des edges qui concernenet un forum identifié par forum_id
        new_n_con_edges.append([rn_id, edge_id])
    
    # Composed of nodes (forum + all members) 
    new_n_con_nodes.append([rn_id, forum_id])
    for person_id in group['nd']: # ici nd sont les ids des personnes (member) qui concernent un forum identifié par forum_id
        new_n_con_nodes.append([rn_id, person_id])
    
    # Node property: nbMember
    nb_members = len(group)
    new_node_props.append([rn_id, "nbMember", nb_members])

# Fonction  pour formater mess commande Cozo 
def format_cozo(rel, rows, cols):
    entries = ",".join(
        "[" + ",".join(f'"{x}"' if isinstance(x, str) else str(x) for x in row) + "]"
        for row in rows
    )
    return f"?[{','.join(cols)}] <- [{entries}] :put {rel} {{{','.join(cols)}}}"

# Chunked insertion 
def chunkify(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

def run_chunks(rel, rows, cols):
    total = ceil(len(rows) / CHUNK_SIZE)
    for i, chunk in enumerate(chunkify(rows, CHUNK_SIZE), 1):
        print(f"[{rel}] chunk {i}/{total}")
        client.run(format_cozo(rel, chunk, cols))

#  Exécuter l'insertion 
run_chunks("reified_node", new_reified_nodes, ["id_rn"])
run_chunks("node_label", new_node_labels, ["id_n","ln"])
run_chunks("n_composed_of_node", new_n_con_nodes, ["id_rn","id_n"])
run_chunks("n_composed_of_edge", new_n_con_edges, ["id_rn","id_e"])
run_chunks("node_prop", new_node_props, ["id_n","key","value"])

print("\n voilà ! Les reified nodes forumMembership ont été créés avec succès")


[reified_node] chunk 1/8
[reified_node] chunk 2/8
[reified_node] chunk 3/8
[reified_node] chunk 4/8
[reified_node] chunk 5/8
[reified_node] chunk 6/8
[reified_node] chunk 7/8
[reified_node] chunk 8/8
[node_label] chunk 1/8
[node_label] chunk 2/8
[node_label] chunk 3/8
[node_label] chunk 4/8
[node_label] chunk 5/8
[node_label] chunk 6/8
[node_label] chunk 7/8
[node_label] chunk 8/8
[n_composed_of_node] chunk 1/170
[n_composed_of_node] chunk 2/170
[n_composed_of_node] chunk 3/170
[n_composed_of_node] chunk 4/170
[n_composed_of_node] chunk 5/170
[n_composed_of_node] chunk 6/170
[n_composed_of_node] chunk 7/170
[n_composed_of_node] chunk 8/170
[n_composed_of_node] chunk 9/170
[n_composed_of_node] chunk 10/170
[n_composed_of_node] chunk 11/170
[n_composed_of_node] chunk 12/170
[n_composed_of_node] chunk 13/170
[n_composed_of_node] chunk 14/170
[n_composed_of_node] chunk 15/170
[n_composed_of_node] chunk 16/170
[n_composed_of_node] chunk 17/170
[n_composed_of_node] chunk 18/170
[n_composed_o

In [3]:
res = client.run("?[n, l]:= *node_label[n, l], l='forumMembership'")
res

,n,l
0,35184376285345,forumMembership
1,35184376285346,forumMembership
2,35184376285347,forumMembership
3,35184376285348,forumMembership
4,35184376285349,forumMembership
...,...,...
79465,35184376364810,forumMembership
79466,35184376364811,forumMembership
79467,35184376364812,forumMembership
79468,35184376364813,forumMembership


In [10]:
res = client.run("?[n, val]:= *node_prop[n, 'nbMember',val], :order -val")
res

,n,val
0,35184372518647,340
1,35184372522503,338
2,35184372511777,269
3,35184372517688,256
4,35184372512372,230
...,...,...
11072,35184372522536,1
11073,35184372522539,1
11074,35184372522542,1
11075,35184372522543,1


In [4]:
#query : 
q1=""" 
# Filtrer les forumMembership avec nbMember > 3
filtered_forum[n_rn] :=
    *node_prop[n_rn, 'nbMember', nb],
    nb > 150

# Récupérer les personnes qui font partie du forumMembership
forum_member[n_rn, person] :=
    filtered_forum[n_rn],
    *n_composed_of_node[n_rn, person],
    *node_label[person, 'person']

# Récupérer leur locationIP
member_ip[person, ip] :=
    forum_member[n_rn, person],
    *node_prop[person, 'locationIP', ip]

# Résultat final
?[person, ip] := member_ip[person, ip]

"""
# print(client.run(q1))


In [5]:
#query : 
q2=""" 
# Filtrer les forumMembership avec nbMember > 3
filtered_forum[n_rn] :=
    *node_prop[n_rn, 'nbMember', nb],
    nb <150

# Récupérer les personnes qui font partie du forumMembership
forum_member[n_rn, person] :=
    filtered_forum[n_rn],
    *n_composed_of_node[n_rn, person],
    *node_label[person, 'person']

# Récupérer leur locationIP
member_ip[person, ip] :=
    forum_member[n_rn, person],
    *node_prop[person, 'locationIP', ip]

# Résultat final
?[person, ip] := member_ip[person, ip]

"""
# print(client.run(q2))

In [7]:
import time
# exécuter 10 fois auto 
n = 10
times = []

for _ in range(n):
    start = time.time()  
    client.run(q2)
    end = time.time()  
    times.append((end - start) * 1000)  # en ms

mean = sum(times) / n
print(f"Temps moyen sur {n} exécutions : {mean:.3f} ms")




Temps moyen sur 10 exécutions : 15280.674 ms
